<a href="https://colab.research.google.com/github/cursuri-inf/IPAP/blob/main/saptamana-03/L3_bucla_de_antrenare.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Caietul „Bucla de antrenare” (Laborator 3)

Completezi cele patru rânduri care lipsesc din bucla de antrenare de la cursul 3 și o pui la lucru pe pinguini: regresia liniară care prezice masa după lungimea aripii. Alegi apoi rata de învățare după curba pierderii și citești ce spune dreapta găsită. Nivelul de bază (secțiunile 0–4) se termină în laborator.

Pașii:

1. Deschizi caietul cu butonul „Open in Colab” de mai sus. La prima rulare, Colab avertizează că documentul nu a fost creat de Google: caietul vine din depozitul cursului, așa că alegi „Run anyway”.
2. Înainte să scrii ceva: File → Save a copy in Drive; lucrezi în copie.
3. Lucrezi în pereche cu un coleg din echipă. Pilotul rulează și scrie, navigatorul citește, prezice și verifică; schimbați rolurile la fiecare secțiune. Într-o echipă de 3, al treilea coleg este verificator: rulează în copia lui și compară rezultatele. Fiecare predă caietul lui.
4. Rulezi celulele în ordine, cu Shift+Enter. La fiecare „Exercițiul B…” te oprești și scrii răspunsul în celula „Răspunsul tău:” de sub el sau completezi codul, acolo unde vezi `...` sau `None`.
5. La final: Runtime → Restart session and run all (rularea de la zero), verifici că ultima celulă de cod afișează „Caietul a rulat până la capăt”, completezi „Declarația AI”, descarci caietul (File → Download → Download .ipynb) și îl încarci la activitatea L3 din Campus Virtual.

Tot caietul rulează pe procesor (CPU), în mai puțin de un minut; nu ai nevoie de GPU. Extensiile E1–E3 sunt opționale, pentru nota 8–10.

Completează:

- Numele tău: …
- Colegul de pereche: …
- Echipa: …

## Harta caietului

Fiecare secțiune pune la lucru un capitol din suportul cursului 3.

| Secțiunea | Capitolul din suportul cursului 3 |
|---|---|
| 1. Datele și tensorii | 2, Tensori; 5.2, Standardizarea |
| 2. Bucla de antrenare | 3.5, Autograd; 5.3, Bucla în PyTorch |
| 3. Rata de învățare | 4, Rata de învățare; 5.3, „Încercați acum” |
| 4. Ce spune dreapta | 5.1, Aceeași dreaptă ca în cursul 2 |
| Extensia E2 | 3.5, Autograd (pasul scris de mână) |

## 0. Pornire

Încarci bibliotecile, verifici versiunile și citești datele: cei 342 de pinguini cu toate măsurătorile, ca în cursurile 2 și 3. Dacă adresa datelor nu răspunde, descarci fișierul `penguins.csv` din Campus Virtual (Cursul 2, „Datele cu pinguini”), îl încarci în panoul Files din bara din stânga și rulezi din nou celula cu datele: caietul îl folosește pe acela.

In [ ]:
import os, sys, time
import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.optim import SGD
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt

T_START = time.time()  # pentru durata totală, afișată la final

print("Python ", sys.version.split()[0])
print("PyTorch", torch.__version__)


def procent(x):
    """0.9394 -> '93,9%' (cu virgulă zecimală)."""
    return f"{100 * x:.1f}%".replace(".", ",")


def numar(x):
    """153500.4 -> '153.500' (punct la mii)."""
    return f"{x:,.0f}".replace(",", ".")


def r(x, zecimale=1):
    """Rotunjire fără „-0.0”."""
    return round(x, zecimale) + 0.0


# culorile graficelor cursului
TEXT, SECUNDAR, ACCENT, SERIE2 = "#161616", "#4A4A4A", "#9B1B30", "#4E6375"
plt.rcParams.update({"axes.spines.top": False, "axes.spines.right": False, "axes.grid": False,
                     "text.color": TEXT, "axes.labelcolor": SECUNDAR,
                     "xtick.color": SECUNDAR, "ytick.color": SECUNDAR})

In [ ]:
URL = "https://raw.githubusercontent.com/mwaskom/seaborn-data/master/penguins.csv"
masuri = ["bill_length_mm", "bill_depth_mm", "flipper_length_mm", "body_mass_g"]

sursa = "penguins.csv" if os.path.exists("penguins.csv") else URL
df = pd.read_csv(sursa).dropna(subset=masuri)  # 342 de pinguini cu toate măsurătorile
print("Pinguini:", len(df))
print(df["species"].value_counts().to_string())

## 1. Datele și tensorii

Aceeași întrebare ca la sfârșitul cursului 2 și în cursul 3: cât cântărește un pinguin, după lungimea aripii? Aceeași împărțire: 80% din pinguini pentru antrenare, 20% pentru test, cu `random_state=0`. Din tabelul pandas faci tensori cu o singură coloană, cu forma `[n, 1]`: n pinguini, câte o măsurătoare pentru fiecare.

In [ ]:
Xr_ant, Xr_test, yr_ant, yr_test = train_test_split(
    df[["flipper_length_mm"]], df["body_mass_g"],
    test_size=0.2, random_state=0)


def coloana(t):
    """Din pandas: tensor cu o singură coloană, forma [n, 1]."""
    return torch.tensor(t.values).float().reshape(-1, 1)


x_ant, y_ant = coloana(Xr_ant), coloana(yr_ant)
x_test, y_test = coloana(Xr_test), coloana(yr_test)
print("x_ant:", x_ant.shape, "| x_test:", x_test.shape)

medie, abatere = x_ant.mean(), x_ant.std()  # doar din antrenare
print("aripa la antrenare: media", r(medie.item()), "mm, abaterea", r(abatere.item()), "mm")

### Exercițiul B1 (prezice ce afișează)

Celula de cod de după răspuns standardizează aripa, ca în cursul 3 (§5.2): `z = (x - medie) / abatere`, cu media și abaterea de la antrenare, aplicate atât pinguinilor de antrenare, cât și celor de test. Apoi afișează forma lui `z_ant`, media și abaterea lui `z_ant` și media și abaterea lui `z_test`.

Înainte s-o rulezi, scrie ce crezi că va afișa: ce formă are `z_ant`? Cât sunt media și abaterea pe antrenare? Dar pe test: tot exact 0 și 1? De ce?

Răspunsul tău: (predicția, scrisă înainte de rulare; după rulare, o frază: ai avut dreptate?)

In [ ]:
z_ant = (x_ant - medie) / abatere
z_test = (x_test - medie) / abatere
print("z_ant:", z_ant.shape)
print("antrenare: media", r(z_ant.mean().item(), 3), "| abaterea", r(z_ant.std().item(), 3))
print("test:      media", r(z_test.mean().item(), 3), "| abaterea", r(z_test.std().item(), 3))

## 2. Bucla de antrenare

Modelul este cel din cursul 3: o dreaptă, masa = w × z + b, adică un strat `nn.Linear(1, 1)` cu doi parametri. Pierderea este cea pătratică medie (`nn.MSELoss`), optimizatorul este SGD, iar pinguinii de antrenare se citesc în loturi de câte 32, în altă ordine la fiecare epocă.

### Exercițiul B2 (completează)

Funcția `pas` face un pas al buclei, pe un lot de pinguini. Primul rând, predicția, este scris. În locul celor patru `...` scrii rândurile care lipsesc, în ordinea din cursul 3 (§5.3): măsori pierderea, ștergi gradientul vechi, calculezi gradientul, muți parametrii. Folosești numele din funcție: `model`, `pierdere_f`, `opt`, `zb`, `yb`.

Apoi rulezi celulele până la verificare. Cât timp rândurile lipsesc, caietul rulează, dar modelul nu învață nimic; verificarea îți spune asta. După ce schimbi funcția, rulezi din nou toate celulele de la ea încoace (Runtime → Run after, din celula cu `pas`).

In [ ]:
# Exercițiul B2: înlocuiește fiecare ... cu rândul care lipsește
def pas(model, pierdere_f, opt, zb, yb):
    """Un pas al buclei, pe un lot de pinguini."""
    predictie = model(zb)    # 1. prezice
    pierdere = ...           # 2. măsoară pierderea dintre predictie și yb
    ...                      # 3. șterge gradientul vechi
    ...                      # 4. calculează gradientul
    ...                      # 5. mută parametrii

Restul buclei este scris: funcția `antreneaza` trece de mai multe ori (epoci) prin toate loturile și cheamă `pas` pe fiecare lot. După fiecare epocă măsoară pierderea pe toți pinguinii de antrenare, fără gradient; prima valoare din istoric este pierderea de dinainte de antrenare. Funcția `regresie` pornește de fiecare dată de la aceiași parametri și cu aceeași ordine a loturilor (`torch.manual_seed(0)`), ca rezultatele să se poată compara.

In [ ]:
loturi = DataLoader(TensorDataset(z_ant, y_ant), batch_size=32, shuffle=True)


def antreneaza(model, pierdere_f, opt, loturi, epoci):
    """Bucla: la fiecare epocă, câte un pas pe fiecare lot.
    Întoarce pierderea pe toată antrenarea: înainte de prima epocă și după fiecare epocă."""
    z, y = loturi.dataset.tensors
    istoric = []
    for epoca in range(epoci + 1):
        if epoca > 0:
            for zb, yb in loturi:
                pas(model, pierdere_f, opt, zb, yb)
        with torch.no_grad():  # doar măsurăm, fără gradient
            istoric.append(pierdere_f(model(z), y).item())
    return istoric


def regresie(rata, epoci):
    """Un model nou, antrenat cu rata de învățare și numărul de epoci date."""
    torch.manual_seed(0)  # aceiași parametri de pornire, aceeași ordine a loturilor
    model = nn.Linear(1, 1)
    opt = SGD(model.parameters(), lr=rata)
    istoric = antreneaza(model, nn.MSELoss(), opt, loturi, epoci)
    return model, istoric


def in_grame(model):
    """w și b înapoi în unitățile pinguinilor: masa ≈ w × aripa + b."""
    w = model.weight.item() / abatere.item()
    b = model.bias.item() - w * medie.item()
    return w, b


def eroare_test(model):
    """Eroarea absolută medie pe pinguinii de test, în grame."""
    with torch.no_grad():
        return (model(z_test) - y_test).abs().mean().item()


print("loturi pe epocă:", len(loturi))

Antrenarea de bază, cu valorile din cursul 3: rata de învățare 0,02 și 50 de epoci. Referința este cea din cursul 2: pentru orice pinguin spunem masa medie de la antrenare.

In [ ]:
RATA, EPOCI = 0.02, 50  # valorile din cursul 3
model, istoric = regresie(RATA, EPOCI)
w, b = in_grame(model)
EROARE_G = eroare_test(model)
REFERINTA_G = (yr_test - yr_ant.mean()).abs().mean()  # mereu masa medie de la antrenare

print("pași în total:", len(loturi) * EPOCI)
print("pierderea:", numar(istoric[0]), "la început,", numar(istoric[-1]), "la final")
print("masa ≈", r(w), "× aripa", "-" if b < 0 else "+", numar(abs(b)))
print("eroarea pe test:", round(EROARE_G), "g | referința:", round(REFERINTA_G), "g")

# verificarea exercițiului B2
if istoric[0] == istoric[-1]:
    print("\nB2: nu încă. Pierderea nu s-a schimbat deloc: modelul nu a învățat nimic.",
          "Completează rândurile din pas(), în ordinea din curs, și rulează din nou celulele de la pas() încoace.")
elif abs(w - 49.4) < 1 and EROARE_G < 320:
    print("\nB2: corect. Bucla ta găsește dreapta din cursul 3: aproximativ 49,4 g/mm, eroarea 314 g.")
else:
    print("\nB2: nu încă. Modelul se schimbă, dar nu ajunge la dreapta din cursul 3.",
          "Verifică cele patru rânduri și ordinea lor (gradientul vechi se șterge la fiecare lot)",
          "și că RATA și EPOCI au valorile 0.02 și 50.")

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 3.8))
ax1.plot(range(len(istoric)), istoric, color=ACCENT, lw=2)
ax1.set_yscale("log")
ax1.set_title("pierderea pe antrenare (scară logaritmică)", loc="left")
ax1.set_xlabel("epoca")
ax2.scatter(x_test, y_test, s=14, color=SECUNDAR, alpha=0.7)
aripi = torch.linspace(x_test.min().item(), x_test.max().item(), 2)
ax2.plot(aripi, w * aripi + b, color=ACCENT, lw=2.5)
ax2.set_title("pinguinii de test și dreapta învățată", loc="left")
ax2.set_xlabel("lungimea aripii (mm)")
ax2.set_ylabel("masa (g)")
plt.tight_layout()
plt.show()

## 3. Rata de învățare

Regula din cursul 3 (§4.2): rata de învățare se alege după curba pierderii. Celula de mai jos antrenează același model cu patru rate, câte 20 de epoci, pornind de fiecare dată de la aceiași parametri. Graficul din stânga are axa pierderii logaritmică (fiecare diviziune înseamnă de 10 ori mai mult) și arată toate cele patru curbe; cel din dreapta mărește ultimele epoci pentru ratele 0,02 și 0,5.

In [ ]:
RATE = [0.001, 0.02, 0.5, 1.5]
curbe = {}
for rata in RATE:
    m, ist = regresie(rata, 20)
    curbe[rata] = ist
    if np.isfinite(ist[-1]):
        w_r, b_r = in_grame(m)
        print(f"rata {rata:<5}  pierderea finală {numar(ist[-1]):>10}  w {r(w_r):5.1f}  "
              f"eroarea pe test {eroare_test(m):5.0f} g")
    else:
        print(f"rata {rata:<5}  pierderea explodează: {numar(ist[1])} după prima epocă, "
              f"apoi {ist[-1]} (prea mare ca să mai fie un număr)")

In [ ]:
culori = {0.001: SERIE2, 0.02: ACCENT, 0.5: TEXT, 1.5: SECUNDAR}
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))
for rata, ist in curbe.items():
    valori = np.nan_to_num(np.array(ist), nan=1e30, posinf=1e30)
    eticheta = "rata " + str(rata).replace(".", ",")
    ax1.plot(range(len(ist)), valori, color=culori[rata], lw=2, marker="o", ms=3)
    deplasare = {0.02: 0.7, 0.5: 1.45}.get(rata, 1)  # etichetele 0,02 și 0,5 nu se suprapun
    if np.isfinite(ist[-1]):
        ax1.text(20.5, ist[-1] * deplasare, eticheta, color=culori[rata], va="center")
    else:
        ax1.text(1.2, 6e8, eticheta + ": urcă", color=culori[rata], va="center")
    if rata in (0.02, 0.5):
        ax2.plot(range(5, 21), np.array(ist[5:]) / 1000, color=culori[rata], lw=2, marker="o", ms=3)
        ax2.text(20.3, ist[-1] / 1000 + (12 if rata == 0.5 else -8), eticheta, color=culori[rata], va="center")
ax1.set_yscale("log")
ax1.set_ylim(5e4, 1e9)
ax1.set_xlim(-0.5, 24)
ax1.set_title("pierderea, toate ratele (scară logaritmică)", loc="left")
ax1.set_xlabel("epoca")
ax2.set_xlim(4.5, 22.5)
ax2.set_title("epocile 5–20 (pierderea în mii)", loc="left")
ax2.set_xlabel("epoca")
plt.tight_layout()
plt.show()

### Exercițiul B3 (modifică și notează)

1. Completează tabelul de mai jos după grafice și după rezultatele afișate: forma curbei (coboară repede și se așază; abia coboară; urcă) și ce faci cu rata (o păstrezi, o crești, o micșorezi).
2. Alege o rată a ta, alta decât cele patru: în celula de cod de după tabel înlocuiești `None` cu rata ta și rulezi. Celula o compară cu rata 0,02. Adaugi rezultatul în ultimul rând al tabelului.
3. Ce rată ai alege pentru acest model și de ce? (2–3 fraze; uită-te și la cât de repede coboară curba, și la locul unde se așază.)

Răspunsul tău:

| Rata | Forma curbei | Ce faci cu rata |
|---|---|---|
| 0,001 | | |
| 0,02 | | |
| 0,5 | | |
| 1,5 | | |
| rata ta: … | | |

Ce rată alegi și de ce:

In [ ]:
# Exercițiul B3: înlocuiește None cu rata ta, de exemplu 0.005
rata_ta = None

if rata_ta is None:
    print("B3: nu ai ales încă rata_ta.")
else:
    m, ist = regresie(rata_ta, 20)
    if np.isfinite(ist[-1]):
        w_r, b_r = in_grame(m)
        print(f"rata ta {rata_ta}: pierderea finală {numar(ist[-1])}, w {w_r:.1f}, "
              f"eroarea pe test {eroare_test(m):.0f} g")
    else:
        print(f"rata ta {rata_ta}: pierderea explodează")
    print(f"rata 0.02: pierderea finală {numar(curbe[0.02][-1])}")
    plt.figure(figsize=(6, 3.5))
    plt.plot(np.nan_to_num(np.array(ist), nan=1e30, posinf=1e30), color=ACCENT, lw=2, label="rata ta")
    plt.plot(curbe[0.02], color=SECUNDAR, lw=1.5, ls="--", label="rata 0,02")
    plt.yscale("log")
    plt.ylim(5e4, 1e9)
    plt.xlabel("epoca")
    plt.gca().xaxis.set_major_locator(plt.MaxNLocator(integer=True))
    plt.legend(frameon=False)
    plt.show()

## 4. Ce spune dreapta

Un model antrenat se citește în unitățile datelor. Dreapta din secțiunea 2 spune cu câte grame crește, în medie, masa unui pinguin pentru fiecare milimetru de aripă. Celula de mai jos afișează intervalul aripilor pe care le-a văzut modelul la antrenare, câteva predicții și eroarea lui lângă referința simplă.

### Exercițiul B4 (interpretează)

Răspunde întâi la punctele 1 și 2 cu creionul, din dreapta afișată în secțiunea 2, apoi rulează celula de după răspuns și verifică.

1. Ce înseamnă, în cuvinte, panta dreptei (aproximativ 49,4 g/mm)?
2. Cât cântărește, după model, un pinguin cu aripa de 210 mm?
3. Modelul prezice și pentru o aripă de 150 mm. De ce nu ai folosi această predicție? Uită-te la intervalul aripilor de la antrenare.
4. Eroarea pe test este de aproximativ 314 g, iar a referinței (mereu masa medie) de aproximativ 619 g. Ce spun cele două cifre, puse una lângă alta, despre cât ajută lungimea aripii? Ai folosi modelul în locul unui cântar? (2–3 fraze)

Răspunsul tău:

In [ ]:
aripi_ant = Xr_ant["flipper_length_mm"]
print("aripile de la antrenare: între", int(aripi_ant.min()), "și", int(aripi_ant.max()), "mm")
print("masa ≈", r(w), "× aripa", "-" if b < 0 else "+", numar(abs(b)))
for aripa in (190, 200, 210, 150):
    print(f"aripa {aripa} mm -> masa prezisă {numar(w * aripa + b)} g")
print("masa medie la antrenare:", numar(yr_ant.mean()), "g")
print("eroarea pe test:", round(EROARE_G), "g | referința:", round(REFERINTA_G), "g")

## Extensii (opționale, pentru nota 8–10)

Alegi cel puțin una. Fiecare are un comutator: îl pui pe `True`, rulezi celula și scrii, în celula „Comentariul tău” de sub ea, ce ai schimbat, ce rezultat ai obținut față de modelul de bază și cum îl explici. Extensia pe care o predai rămâne cu comutatorul pe `True`, ca rezultatele ei să se vadă în caiet. Toate rulează pe procesor, în câteva secunde. Rulează extensiile după ce nivelul de bază este gata: toate folosesc funcția ta `pas` sau rezultatele din secțiunile 1–3.

### Extensia E1: masa după trei măsurători

Masa după lungimea aripii și după lungimea și adâncimea ciocului: un strat `nn.Linear(3, 1)`, cu cele trei coloane standardizate pe antrenare, aceeași împărțire 80/20, aceeași rată și același număr de epoci ca în secțiunea 2. Compară eroarea pe test cu modelul cu o singură măsurătoare și citește ponderile: care măsurătoare contează cel mai mult? De ce se pot compara ponderile între ele doar pentru că intrările sunt standardizate?

In [ ]:
RULEAZA_E1 = False  # True: rulezi extensia

if RULEAZA_E1:
    col_e2 = ["flipper_length_mm", "bill_length_mm", "bill_depth_mm"]
    Xa2, Xt2, ya2, yt2 = train_test_split(df[col_e2], df["body_mass_g"],
                                          test_size=0.2, random_state=0)  # aceiași pinguini
    m2, s2 = Xa2.mean(), Xa2.std()
    za2 = torch.tensor(((Xa2 - m2) / s2).values).float()
    zt2 = torch.tensor(((Xt2 - m2) / s2).values).float()
    torch.manual_seed(0)
    model_e2 = nn.Linear(3, 1)
    opt2 = SGD(model_e2.parameters(), lr=RATA)
    loturi2 = DataLoader(TensorDataset(za2, coloana(ya2)), batch_size=32, shuffle=True)
    antreneaza(model_e2, nn.MSELoss(), opt2, loturi2, EPOCI)
    with torch.no_grad():
        eroare_e2 = (model_e2(zt2) - coloana(yt2)).abs().mean().item()
    print("eroarea pe test: trei măsurători", round(eroare_e2), "g | doar aripa", round(EROARE_G), "g")
    print("ponderile (grame pentru o abatere standard în plus):")
    for c, pondere in zip(col_e2, model_e2.weight[0].tolist()):
        print(f"  {c:18} {pondere:7.1f}")
else:
    print("E1 nu rulează (RULEAZA_E1 = False).")

Comentariul tău (E1):

### Extensia E2: pasul scris de mână, fără optimizator

Optimizatorul SGD face, pentru fiecare parametru p, un singur lucru: p ← p − rata × gradientul lui p. Scrii singur acest pas, ca în demonstrația cu autograd din cursul 3 (§3.5), în locul lui `opt.step()` și `opt.zero_grad()`: completezi rândul cu `...` din funcția `pas_de_mana`. Antrenezi apoi regresia liniară din secțiunea 2 și compari w, b și eroarea pe test cu cele obținute cu SGD. Ce observi și de ce?

In [ ]:
RULEAZA_E2 = False  # True: rulezi extensia

if RULEAZA_E2:
    def pas_de_mana(model, pierdere_f, rata, zb, yb):
        """Același pas ca pas(), fără optimizator."""
        pierdere = pierdere_f(model(zb), yb)
        pierdere.backward()
        with torch.no_grad():  # pasul nu face parte din calculul urmărit
            for p in model.parameters():
                p -= ...           # completează: rata înmulțită cu gradientul lui p
                p.grad.zero_()     # ștergi gradientul, pentru lotul următor

    torch.manual_seed(0)  # aceiași parametri de pornire, aceeași ordine a loturilor
    model_e3 = nn.Linear(1, 1)
    for epoca in range(EPOCI):
        for zb, yb in loturi:
            pas_de_mana(model_e3, nn.MSELoss(), RATA, zb, yb)
    w3, b3 = in_grame(model_e3)
    print("de mână: w", r(w3, 3), "| b", r(b3, 2), "| eroarea", r(eroare_test(model_e3), 2), "g")
    print("cu SGD:  w", r(w, 3), "| b", r(b, 2), "| eroarea", r(EROARE_G, 2), "g")
else:
    print("E2 nu rulează (RULEAZA_E2 = False).")

Comentariul tău (E2):

### Extensia E3: datele echipei

Aceeași buclă, pe un tabel ales de echipa ta: un fișier CSV cu o coloană numerică de prezis (ținta) și 1–3 coloane numerice de intrare, ca la pinguini: o regresie. Folosești doar date publice (de exemplu de pe data.gov.ro sau Kaggle) sau date generate de voi; nu folosești date personale și nici date interne de la locul de muncă.

Încarci fișierul în panoul Files din bara din stânga sau pui `INCARCA_CSV = True` și îl alegi. Completezi numele fișierului, ținta și intrările. Codul împarte datele 80/20, standardizează intrările pe antrenare, antrenează 50 de epoci cu funcția ta `pas` și compară eroarea cu referința simplă, media țintei de la antrenare. Dacă pierderea urcă, micșorezi `RATA_E3` (exercițiul B3). După ce ai încărcat fișierul, pui înapoi `INCARCA_CSV = False`: altfel, la „Run all”, caietul se oprește și așteaptă un fișier.

In [ ]:
INCARCA_CSV = False             # True: alegi fișierul de pe calculator (doar în Colab)
FISIER_E3 = "datele_noastre.csv"
TINTA = "..."                   # coloana de prezis
INTRARI = ["..."]               # 1–3 coloane numerice
RATA_E3 = 0.02

if INCARCA_CSV:
    from google.colab import files  # există doar în Colab
    files.upload()

if os.path.exists(FISIER_E3):
    d3 = pd.read_csv(FISIER_E3).dropna(subset=INTRARI + [TINTA])
    y3 = d3[TINTA].astype(float)
    Xa3, Xt3, ya3, yt3 = train_test_split(d3[INTRARI].astype(float), y3, test_size=0.2, random_state=0)
    m3, s3 = Xa3.mean(), Xa3.std()
    za3 = torch.tensor(((Xa3 - m3) / s3).values).float()
    zt3 = torch.tensor(((Xt3 - m3) / s3).values).float()
    ta3, tt3 = coloana(ya3), coloana(yt3)
    torch.manual_seed(0)
    model_echipa = nn.Linear(len(INTRARI), 1)
    loturi3 = DataLoader(TensorDataset(za3, ta3), batch_size=32, shuffle=True)
    ist3 = antreneaza(model_echipa, nn.MSELoss(), SGD(model_echipa.parameters(), lr=RATA_E3), loturi3, 50)
    with torch.no_grad():
        iesire = model_echipa(zt3)
    print("rânduri:", len(d3), "| antrenare", len(Xa3), "| test", len(Xt3))
    print("eroarea absolută medie pe test:", r((iesire - tt3).abs().mean().item(), 2))
    print("referința (media de la antrenare):", r((tt3 - ta3.mean()).abs().mean().item(), 2))
    plt.figure(figsize=(6, 3.2))
    plt.plot(np.nan_to_num(np.array(ist3), nan=1e30, posinf=1e30), color=ACCENT, lw=2)
    plt.yscale("log")
    plt.title("pierderea pe antrenare", loc="left")
    plt.xlabel("epoca")
    plt.show()
else:
    print(f"E3 nu rulează: fișierul {FISIER_E3} nu există.")

Comentariul tău (E3): ce date ați ales și de unde, ce ați prezis, ce rezultat ați obținut față de referința simplă și ce rată ați folosit.

## Verificarea finală

Celula de mai jos are sens după Runtime → Restart session and run all: dacă afișează mesajul, caietul rulează de la început până la sfârșit fără erori.

In [ ]:
durata = time.time() - T_START
print(f"Caietul a rulat până la capăt (durata: {durata:.0f} s).")
print("Regresia: eroarea pe test", round(EROARE_G), "g | referința", round(REFERINTA_G), "g")
if istoric[0] == istoric[-1]:
    print("Atenție: funcția pas() nu este completă (exercițiul B2).")

## Ce ai învățat

- Bucla de antrenare are patru rânduri care se repetă pe fiecare lot: pierderea, ștergerea gradientului vechi, gradientul, pasul.
- Rata de învățare se alege după curba pierderii: dacă abia coboară, o crești; dacă urcă, o micșorezi. Rata care coboară cel mai repede nu ajunge neapărat cel mai jos.
- Un model antrenat se citește în unitățile datelor (aici, grame pe milimetru) și se folosește doar în intervalul pe care l-a văzut la antrenare.
- Eroarea unui model are sens doar lângă referința simplă.

## Legătura cu cursul 4

La cursul 4, aceeași buclă antrenează un clasificator: regresia logistică, care desparte pinguinii Adelie de Chinstrap, apoi o rețea cu mai mulți neuroni în straturi, care poate trasa granițe curbe. Funcția ta `pas` rămâne exact aceeași.

## Declarație AI

Completezi la fiecare caiet predat, la persoana întâi:

- Instrumentul AI: …
- Pentru ce l-am folosit: …
- Ce am verificat singur (și cum): …

Dacă nu ai folosit niciun instrument AI, înlocuiești cele trei rânduri cu: Nu am folosit asistenți AI.